# BITS Pilani — Derivatives & Risk Management
## Lab Assignment: Quantitative Risk Profiling of an Underlying Asset using `yfinance`
### STUDENT STARTER NOTEBOOK

**Marks: 15**  |  Full task descriptions, the personalization rules, the banned-function list, and the
grading rubric are in the assignment brief (`DRM_yfinance_Lab_Assignment.docx`) — **read that document first.**
This notebook only repeats the formulas and gives you the boilerplate/plumbing so you don't waste time on
API syntax; it does **not** repeat the full task instructions or grading criteria.

**How to use this notebook:**

- Cells marked `# TODO` are yours to complete. Hints are given as comments; the formulas are in the markdown
  cells above each task, matching the assignment brief.
- Everything else (imports, the ticker list, the data-download plumbing, plot scaffolding) is given so a
  fresh "Run All" executes cleanly top to bottom even before you've filled anything in — TODO variables are
  set to `None` until you implement them, so you can check your progress incrementally instead of hitting a
  crash on the first empty cell.
- Re-read Section 5 of the assignment brief (Academic Integrity Policy) before you start, especially the list
  of banned functions for Task 2 and Task 3 — using them scores that item **zero** even if the number is right.


## Academic Integrity Declaration

Replace the bracketed fields below, keep this as the first cell of your submitted notebook, and delete this
instruction line before you submit.

> I, **[Full Name]**, Roll Number **[Roll Number]**, declare that the code, computations and written analysis
> in this notebook are entirely my own work. I have **not** used ChatGPT, Claude, Gemini, Copilot, or any other
> AI tool to generate, complete, or solve any part of this assignment. I understand that a violation of this
> declaration will result in a mark of zero for this assignment and may be referred under the Institute's
> academic dishonesty policy.
>
> **Signed (typed full name):** _______________ &nbsp;&nbsp; **Date:** _______________


In [1]:
import datetime
print("Notebook executed at:", datetime.datetime.now())


Notebook executed at: 2026-09-03 10:00:35.954043


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import yfinance as yf

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")


## Task 0 — Personalization key & data acquisition  *(1.5 marks)*

**Ticker:** last two digits of your roll number's numeric part, mod 24, index into `TICKER_UNIVERSE` below.

**Date window:** sum of all digits in your roll number × 17, mod 1000 = offset in days from `2018-01-01`.
Window = `[start, start + 730 days]`.

**Benchmark:** everyone uses `^NSEI` (Nifty 50).

See Section 2 of the assignment brief for the full explanation.


In [3]:
TICKER_UNIVERSE = [
    "RELIANCE.NS", "TCS.NS", "INFY.NS", "HDFCBANK.NS", "ICICIBANK.NS", "SBIN.NS",
    "ITC.NS", "HINDUNILVR.NS", "BAJFINANCE.NS", "KOTAKBANK.NS", "LT.NS", "AXISBANK.NS",
    "MARUTI.NS", "SUNPHARMA.NS", "TATAMOTORS.NS", "WIPRO.NS", "ONGC.NS", "NTPC.NS",
    "POWERGRID.NS", "ADANIENT.NS", "ASIANPAINT.NS", "BHARTIARTL.NS", "TITAN.NS", "ULTRACEMCO.NS",
]
BENCHMARK = "^NSEI"

def derive_dataset(roll_number: str, base_start="2018-01-01", window_days=730):
    '''Map a roll number to (ticker, start_date, end_date). See Task 0 above for the rule.

    Steps:
      1. Pull out every digit in `roll_number` (hint: str.isdigit()).
      2. Ticker index = int of the last two digits, mod len(TICKER_UNIVERSE).
      3. Date offset (days) = sum of ALL digits, times 17, mod 1000.
      4. start = base_start + offset_days ; end = start + window_days.
    '''
    # TODO: implement the 4 steps above and return (ticker, start_date, end_date)
    return None, None, None

MY_ROLL_NUMBER = ""   # TODO: fill in your own roll number, e.g. "2021A7PS0083P"
TICKER, START, END = derive_dataset(MY_ROLL_NUMBER)
print(f"Roll number : {MY_ROLL_NUMBER}")
print(f"Assigned ticker : {TICKER}")
print(f"Assigned window : {START}  to  {END}")


Roll number : 
Assigned ticker : None
Assigned window : None  to  None


In [4]:
# Data download -- this cell is given in full; it will start working once
# derive_dataset() above returns real values instead of None.
raw, bench = None, None
if TICKER is not None:
    raw = yf.download(TICKER, start=str(START), end=str(END), auto_adjust=False, progress=False)
    bench = yf.download(BENCHMARK, start=str(START), end=str(END), auto_adjust=False, progress=False)
    print(f"Rows fetched — {TICKER}: {len(raw)},  {BENCHMARK}: {len(bench)}")
else:
    print("Complete derive_dataset() above first, then re-run this cell.")


Complete derive_dataset() above first, then re-run this cell.


In [5]:
# TODO: display the first 5 and last 5 rows of `raw` (two separate outputs / cells is fine)


In [6]:
# TODO: save `raw` and `bench` to CSV -- you will submit these files alongside the notebook.
# Hint: DataFrame.to_csv("<some_filename>.csv")


## Task 1 — Returns  *(1.5 marks)*

$$ r_t^{simple} = \frac{P_t - P_{t-1}}{P_{t-1}} \qquad\qquad r_t^{log} = \ln\left(\frac{P_t}{P_{t-1}}\right) $$

Use the **Adjusted Close** column. Compute both series for your stock and for the benchmark.


In [7]:
price = raw["Adj Close"] if raw is not None else None
bench_price = bench["Adj Close"] if bench is not None else None

# TODO: simple_ret = price.pct_change().dropna()
simple_ret = None

# TODO: log_ret = np.log(price / price.shift(1)).dropna()
log_ret = None

# TODO: bench_log_ret -- same idea, applied to bench_price
bench_log_ret = None


In [8]:
# TODO: plot (a) the price series and (b) the log-return series.
# Hint: fig, ax = plt.subplots(1, 2, figsize=(12, 3.5)) ; ax[0].plot(...) ; ax[1].plot(...)


**Your answer (1 sentence):** why are log returns, not simple returns, the standard input for
derivatives pricing models?

*(write your answer here)*


## Task 2 — Manual Descriptive Statistics  *(3 marks)*

**Reminder — banned for this task:** `df.describe()`, `.mean()/.median()/.std()/.var()/.skew()/.kurt()`,
`np.mean/median/std/var/percentile`, `scipy.stats.describe/skew/kurtosis`. Build every statistic from `sum`,
`sort`, indexing and arithmetic on a NumPy array only. See assignment brief Section 5.1 for the exact penalty.

$$ \bar{x} = \frac{1}{n}\sum_{i=1}^n x_i \qquad
s^2 = \frac{1}{n-1}\sum_{i=1}^n (x_i-\bar{x})^2 \qquad
s = \sqrt{s^2} $$

$$ g_1 = \frac{1}{n}\sum_{i=1}^n \left(\frac{x_i-\bar{x}}{s}\right)^3 \ \text{(skewness)}
\qquad
g_2 = \frac{1}{n}\sum_{i=1}^n \left(\frac{x_i-\bar{x}}{s}\right)^4 - 3 \ \text{(excess kurtosis)} $$


In [9]:
def manual_stats(x):
    x = np.asarray(x, dtype=float)
    n = len(x)

    # TODO 1: mean = sum(x) / n
    mean = None

    # TODO 2: median -- sort x (`xs = sorted(x)`), then take the middle value
    #         (average the two middle values if n is even)
    xs = sorted(x)
    median = None

    # TODO 3: sample variance (n-1 denominator), then std = sqrt(variance)
    variance = None
    std = None

    # TODO 4: skewness = (1/n) * sum( ((v - mean) / std) ** 3  for v in x )
    skewness = None

    # TODO 5: excess kurtosis = (1/n) * sum( ((v - mean) / std) ** 4  for v in x ) - 3
    exc_kurtosis = None

    # TODO 6: Q1 and Q3 -- write your own linear-interpolation quantile() helper
    #         (rank = p * (n - 1); interpolate between xs[floor(rank)] and xs[ceil(rank)])
    #         Do NOT use np.percentile.
    q1 = None
    q3 = None

    return {
        "n": n, "mean": mean, "median": median, "std_dev": std, "variance": variance,
        "min": xs[0], "max": xs[-1], "Q1": q1, "Q3": q3,
        "IQR": (q3 - q1) if (q1 is not None and q3 is not None) else None,
        "skewness": skewness, "excess_kurtosis": exc_kurtosis,
    }

# TODO: once log_ret and bench_log_ret exist (Task 1), build a summary table:
# stats_table = pd.DataFrame({
#     f"{TICKER} log return": manual_stats(log_ret.values),
#     f"{BENCHMARK} log return": manual_stats(bench_log_ret.values),
# }).T
# stats_table


**Your interpretation (2-3 sentences):** what does the sign of skewness and the magnitude of excess
kurtosis tell you about the shape of your stock's return distribution relative to your benchmark's?

*(write your answer here)*


## Task 3 — Risk Metrics for a Derivatives Desk  *(3.5 marks)*

No `pyfolio`/`empyrical`/`quantstats`. `scipy.stats.norm` is allowed (for the parametric VaR z-score).

- **Annualised volatility:** $\sigma_{ann} = \sigma_{daily}\sqrt{252}$
- **Historical VaR** at confidence $c$: the $(1-c)$ empirical percentile of the return distribution
- **Parametric (Gaussian) VaR:** $VaR_c = -(\mu + z_{(1-c)}\,\sigma)$
- **Expected Shortfall / CVaR (95%):** mean of returns at or below the 95% historical VaR threshold
- **Maximum drawdown:** largest peak-to-trough decline in the cumulative price path
- **Sharpe ratio:** $\dfrac{\bar{r}_{ann} - r_f}{\sigma_{ann}}$  (state your $r_f$ source)


In [10]:
RF_ANNUAL = None   # TODO: pick and cite a risk-free rate source (e.g. 91-day T-bill yield)

# TODO: mu_d, sig_d = log_ret.mean(), log_ret.std()   -- (np.mean/std ARE allowed outside Task 2)
mu_d, sig_d = None, None

# TODO: ann_vol = sig_d * np.sqrt(252) ; ann_ret = mu_d * 252
ann_vol, ann_ret = None, None

# TODO: hist_var_95 = -np.percentile(log_ret.values, 5)  ; hist_var_99 similarly at the 1st percentile
hist_var_95, hist_var_99 = None, None

# TODO: parametric VaR using stats.norm.ppf(0.95) / .ppf(0.99) as z-scores
param_var_95, param_var_99 = None, None

# TODO: Expected Shortfall (95%) -- mean of the log returns that are <= -hist_var_95
es_95 = None

# TODO: maximum drawdown -- build the cumulative return series, its running max, then
#       drawdown = cum/running_max - 1, and take drawdown.min()
max_dd = None

# TODO: sharpe = (ann_ret - RF_ANNUAL) / ann_vol
sharpe = None

risk_table = pd.Series({
    "Annualised volatility": ann_vol,
    "Annualised mean return": ann_ret,
    "Historical VaR (95%, 1-day)": hist_var_95,
    "Historical VaR (99%, 1-day)": hist_var_99,
    "Parametric VaR (95%, 1-day)": param_var_95,
    "Parametric VaR (99%, 1-day)": param_var_99,
    "Expected Shortfall / CVaR (95%)": es_95,
    "Maximum drawdown": max_dd,
    "Sharpe ratio": sharpe,
}, name=TICKER).to_frame()
risk_table


,0
Annualised volatility,None
Annualised mean return,None
"Historical VaR (95%, 1-day)",None
"Historical VaR (99%, 1-day)",None
"Parametric VaR (95%, 1-day)",None
"Parametric VaR (99%, 1-day)",None
Expected Shortfall / CVaR (95%),None
Maximum drawdown,None
Sharpe ratio,None


In [11]:
# TODO: plot the drawdown series (fill_between reads nicely here)


**Your interpretation (2-3 sentences):** what would a risk desk conclude about margining and tail risk
on this name from your VaR and Expected Shortfall numbers? What does the gap (or lack of gap) between
historical and parametric VaR tell you?

*(write your answer here)*


## Task 4 — Distribution Shape & Option-Pricing Model Assumptions  *(2.5 marks)*

`scipy.stats.jarque_bera` and `scipy.stats.probplot` are allowed for this task.


In [12]:
# TODO: (a) histogram of log_ret with a fitted normal curve overlaid
#           -- x_grid = np.linspace(log_ret.min(), log_ret.max(), 200)
#           -- stats.norm.pdf(x_grid, mu_d, sig_d) gives the fitted curve
# TODO: (b) stats.probplot(log_ret.values, dist="norm", plot=<an axis>) for the Q-Q plot


In [13]:
# TODO: run the Jarque-Bera test
# jb_stat, jb_p = stats.jarque_bera(log_ret.values)
jb_stat, jb_p = None, None
print(f"Jarque-Bera statistic = {jb_stat},  p-value = {jb_p}")
# TODO: print your normality conclusion at the 5% significance level


Jarque-Bera statistic = None,  p-value = None


**Your interpretation (2-3 sentences):** if normality is rejected, what does that imply for a
Black–Scholes price on options written on this stock, and for the volatility skew/smile observed in real
options markets?

*(write your answer here)*


## Task 5 — Market Risk Sensitivity: Beta  *(1.5 marks)*

$$ \beta = \frac{\text{Cov}(r_{stock},\, r_{market})}{\text{Var}(r_{market})} $$


In [14]:
# TODO: align the two return series on their common dates
# aligned = pd.concat([log_ret, bench_log_ret], axis=1, join="inner")
# aligned.columns = ["stock", "market"]
aligned = None

# TODO: beta via covariance -- np.cov(aligned["stock"], aligned["market"]) gives a 2x2 matrix;
#       beta = cov_matrix[0, 1] / cov_matrix[1, 1]
beta = None

# TODO: alpha = aligned["stock"].mean() - beta * aligned["market"].mean()
alpha = None

# TODO: correlation and R-squared
corr, r_squared = None, None

print(f"Beta = {beta}")
print(f"Alpha (daily) = {alpha}")
print(f"R-squared = {r_squared}")


Beta = None
Alpha (daily) = None
R-squared = None


In [15]:
# TODO: scatter plot of stock vs. market daily log returns with the fitted regression line overlaid


**Your interpretation (2-3 sentences):** what does your R² imply about how much of this stock's risk
a Nifty-futures hedge would actually remove, and what is left over?

*(write your answer here)*


## Task 6 — Written Risk Interpretation Report  *(1.5 marks)*

300–500 words, in your own words. **Every number you quote must match a number that appears in your own
output cells above** — this is checked directly against your notebook.

A structure you can use (delete the bracketed hints once you've replaced them with your own numbers and
reasoning):

> Over the assigned window, [TICKER] exhibited an annualised volatility of approximately ___%, [higher/lower]
> than the Nifty 50 benchmark's ___%, consistent with a beta of ___ indicating [amplified/muted] systematic
> exposure. The 95% one-day historical VaR of ___% versus a parametric VaR of ___% shows a [small/material]
> gap, which — combined with the Jarque–Bera result and the skew/kurtosis measured in Task 2 — indicates the
> stock's true tail risk is [understated/roughly matched] by a normal-distribution assumption. For a
> derivatives desk, this implies ___. From a hedging standpoint, an R² of ___% means a Nifty-future hedge
> would remove only that much of this name's return variance, leaving ___% as idiosyncratic risk. Overall,
> [TICKER]'s risk profile suggests [conservative/moderate/aggressive] suitability for ___ strategy, because ___.

*(write your report here)*


## Optional Bonus (+1 mark, not counted toward the 15)

```python
tkr = yf.Ticker(TICKER)
expiries = tkr.options
chain = tkr.option_chain(expiries[0])
calls, puts = chain.calls, chain.puts
atm_iv = calls.iloc[(calls['strike'] - price.iloc[-1]).abs().idxmin()]['impliedVolatility']
print(f"Historical vol (annualised): {ann_vol:.2%}   vs.   ATM implied vol: {atm_iv:.2%}")
```

Compare ATM implied volatility to your Task 3 historical/annualised volatility and discuss the volatility
risk premium and what it means for option buyers vs. sellers.
